In [23]:
import pandas as pd
import re
from itertools import product

class Node:
    """Nodo del árbol lógico."""
    def __init__(self, op=None, children=None, value=None):
        self.op = op          # "AND", "OR" o None
        self.children = children or []
        self.value = value    # asignatura si es hoja

    def __repr__(self):
        if self.op:
            return f"({self.op}: {self.children})"
        else:
            return self.value


def tokenize(expr: str):
    """Convierte string en tokens (paréntesis, operadores, códigos)."""
    expr = expr.replace(" A ", " AND ").replace(" O ", " OR ")
    tokens = re.findall(r'\(|\)|AND|OR|[A-Z0-9]+(?: [0-9]{2})?', expr)
    return tokens


def parse_tokens(tokens):
    """Convierte lista de tokens en árbol lógico usando una pila."""
    def parse_expr(idx=0):
        items = []
        op = None
        while idx < len(tokens):
            tok = tokens[idx]
            if tok == "(":
                node, idx = parse_expr(idx + 1)
                items.append(node)
            elif tok == ")":
                break
            elif tok in ("AND", "OR"):
                op = tok
                idx += 1
                continue
            else:  # asignatura
                items.append(Node(value=tok))
            idx += 1

        # Si no hay operador → único elemento
        if not items:
            return Node(value=""), idx
        if op is None:
            return items[0], idx
        else:
            return Node(op=op, children=items), idx

    node, _ = parse_expr(0)
    return node


def expand(node: Node):
    """Expande el árbol en todas las combinaciones posibles."""
    if node.value:  # hoja
        return [[node.value]]

    if node.op == "AND":
        expanded_children = [expand(c) for c in node.children]
        combos = list(product(*expanded_children))
        return [sum(c, []) for c in combos]

    if node.op == "OR":
        combos = []
        for c in node.children:
            combos.extend(expand(c))
        return combos

    return [[]]


def procesar_prerrequisitos(df_reglas: pd.DataFrame, df_plan: pd.DataFrame) -> pd.DataFrame:
    """
    Procesa las reglas de prerequisitos usando un parser tipo árbol y genera
    columnas separadas con las combinaciones válidas.
    """
    asignaturas_validas = set(df_plan['MatCrso'].astype(str).str.strip().unique())
    resultados = []

    for _, row in df_reglas.iterrows():
        rule = row['prereq completo']
        if pd.isna(rule) or rule.strip() == "":
            resultados.append([])
            continue

        tokens = tokenize(rule)
        tree = parse_tokens(tokens)
        opciones = expand(tree)

        # Quitar duplicados internos y validar
        opciones_limpias = []
        for op in opciones:
            # mantener orden pero sin duplicados
            seen, clean = set(), []
            for m in op:
                if m not in seen:
                    clean.append(m)
                    seen.add(m)
            # validar
            if all(m in asignaturas_validas for m in clean):
                opciones_limpias.append(" & ".join(clean))

        resultados.append(opciones_limpias)

    # Crear dataframe de salida
    max_len = max(len(r) for r in resultados) if resultados else 0
    df_out = df_reglas.copy()
    for i in range(max_len):
        df_out[f'Opcion_Prereq_{i+1}'] = [r[i] if i < len(r) else "" for r in resultados]

    return df_out




In [24]:
def main():
    """
    Función principal para ejecutar el procesamiento
    """
    print("Procesador de Prerrequisitos Universitarios")
    print("=" * 50)
    
    # Opción para mostrar ejemplos o prueba compleja
        
    # Solicitar la ruta del archivo principal
    file_path = 'para trabajar pre req progv2.xlsx'#input("Ingresa la ruta del archivo principal de prerrequisitos (CSV o Excel): ").strip()
    
    if not file_path:
        print("No se proporcionó una ruta de archivo.")
        return None
    
    # Solicitar el archivo de asignaturas válidas
    courses_file_path ='asignaturas en plan de estudio.xlsx' #input("\nIngresa la ruta del archivo 'asignaturas en plan de estudio.xlsx' (o presiona Enter para omitir filtrado): ").strip()
    
    if not courses_file_path:
        print("⚠️ No se proporcionó archivo de asignaturas. Se procesarán todos los prerrequisitos sin filtrar.")
        courses_file_path = None
    
    
    # Procesar el archivo
    print(f"\nProcesando archivo: {file_path}")
    if courses_file_path:
        print(f"Archivo de asignaturas válidas: {courses_file_path}")
    
    df_req= pd.read_excel(file_path) if file_path and str(file_path).endswith(('.xlsx', '.xls')) else pd.read_csv(file_path)
    df_courses= pd.read_excel(courses_file_path) if courses_file_path and str(courses_file_path).endswith(('.xlsx', '.xls')) else pd.read_csv(courses_file_path) if courses_file_path else None

    #df_req= df_req[df_req["Smbarul_Key_Rule"]=="FIS1033"]
    df_result = procesar_prerrequisitos(df_req, df_courses)
    
    # Mostrar información sobre el resultado
    print(f"\n✅ Procesamiento completado!")
    print(f"Total de filas: {len(df_result)}")
    
    # Contar cuántas columnas de opciones se crearon
    option_cols = [col for col in df_result.columns if col.startswith('Opcion_Prereq_')]
    print(f"Columnas de opciones creadas: {len(option_cols)}")
    
    # Contar opciones válidas totales
    valid_options_count = 0
    for col in option_cols:
        valid_options_count += sum(1 for val in df_result[col] if val and str(val).strip())
    print(f"Total de opciones de prerrequisitos válidas: {valid_options_count}")
    
    # Mostrar filas con prerrequisitos (no vacías)
    non_empty_prereqs = df_result[df_result['prereq completo'].notna() & (df_result['prereq completo'] != '')]
    if len(non_empty_prereqs) > 0:
        print(f"\nFilas con prerrequisitos: {len(non_empty_prereqs)}")
        print("\nPrimeras 3 filas con prerrequisitos:")
        cols_to_show = []
        if 'Smbarul_Key_Rule' in df_result.columns:
            cols_to_show.append('Smbarul_Key_Rule')
        cols_to_show.extend(['prereq completo'] + option_cols[:5])  # Mostrar solo las primeras 5 columnas de opciones
        
        available_cols = [col for col in cols_to_show if col in df_result.columns]
        print(non_empty_prereqs[available_cols].head(3).to_string())
    
    # Guardar el resultado
    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
    file_parts = file_path.rsplit('.', 1)
    if len(file_parts) == 2:
        output_path = file_parts[0] +timestamp+ '_procesado_v2.' + file_parts[1]
    else:
        output_path = file_path +timestamp+ '_procesado_v2.csv'
    
    if file_path.endswith('.csv') or not file_path.endswith(('.xlsx', '.xls')):
        df_result.to_csv(output_path, index=False)
    else:
        df_result.to_excel(output_path, index=False)
    
    print(f"\n💾 Archivo guardado como: {output_path}")
    
    return df_result
    
  

if __name__ == "__main__":
    main()

Procesador de Prerrequisitos Universitarios

Procesando archivo: para trabajar pre req progv2.xlsx
Archivo de asignaturas válidas: asignaturas en plan de estudio.xlsx

✅ Procesamiento completado!
Total de filas: 3101
Columnas de opciones creadas: 14
Total de opciones de prerrequisitos válidas: 2572

Filas con prerrequisitos: 2063

Primeras 3 filas con prerrequisitos:
    Smbarul_Key_Rule                                                              prereq completo Opcion_Prereq_1 Opcion_Prereq_2 Opcion_Prereq_3 Opcion_Prereq_4 Opcion_Prereq_5
341          IGL4992                          IGL4990 O HICR 001 O CICR 001 O ECCR 001 O  IGL0251         IGL4990         IGL0251                                                
342          IGL4965                                     IGL4962 O CIRI 001 O HIRI 001 O EXIR 001                                                                                
343          IGL1350   HINI 001 O CINI 001 O EXIN 001 O ECCR 003 O HICR 003 O CICR 003 O  IGL499